# Error Analysis — CARLA PLN

Notebook ini menganalisis **dimana dan mengapa** model CARLA salah pada data PLN.

Urutan analisis:
1. Setup & konfigurasi
2. Jalankan inferensi ulang (per-file, per-window)
3. Score distribution (normal vs anomali)
4. ROC & PR curves
5. Confusion matrix
6. Tabel FP / FN per file
7. Time series — FP files (normal diprediksi anomali)
8. Time series — FN files (pencuri yang lolos)
9. Per-window score heatmap (file tertentu)

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import seaborn as sns
from pathlib import Path
from sklearn.metrics import (
    roc_curve, roc_auc_score,
    precision_recall_curve, average_precision_score,
    confusion_matrix, f1_score
)

from utils.config import create_config
from utils.common_config import get_model, get_val_transformations1, inject_sub_anomaly
from utils.collate import collate_custom
from utils.dataset_channels import get_pln_train_files, get_pln_test_files
from data.PLN import PLN

BASE_DIR = Path('..').resolve()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'BASE_DIR: {BASE_DIR}')

Device: cpu
BASE_DIR: C:\CARLA-20260708-1450


## 1. Konfigurasi

Ganti `AGG` dan `SEED` sesuai model yang ingin dianalisis.

In [3]:
# ============================================================
# KONFIGURASI — sesuaikan di sini
# ============================================================
AGG   = 'mean'   # 'mean' | 'max' | 'pct95'
SEED  = None     # None = default (no suffix) | 0, 1, 2, ... untuk multiseed

CONFIG_ENV = str(BASE_DIR / 'configs' / 'env.yml')
CONFIG_CLS = str(BASE_DIR / 'configs' / 'classification' / 'carla_classification_pln.yml')

p = create_config(CONFIG_ENV, CONFIG_CLS, 'All')
MODEL_PATH = p['classification_model']
print(f'Model path : {MODEL_PATH}')
print(f'Aggregation: {AGG}')
assert Path(MODEL_PATH).exists(), f'Model belum ada: {MODEL_PATH}. Jalankan run_pln.py dahulu.'

Model path : results/pln\All/classification\model.pth.tar
Aggregation: mean


AssertionError: Model belum ada: results/pln\All/classification\model.pth.tar. Jalankan run_pln.py dahulu.

## 2. Load Model & Inferensi Per-File + Per-Window

In [ ]:
# --- Load model ---
model = get_model(p)
model = torch.nn.DataParallel(model)
ckpt  = torch.load(MODEL_PATH, map_location='cpu', weights_only=False)
model.module.load_state_dict(ckpt['model'])
model.to(device).eval()

val_transform = get_val_transformations1(p)
sanomaly      = inject_sub_anomaly(p)

# --- Cari normal_label dari training pool ---
@torch.no_grad()
def get_logits(model, ts):
    if ts.ndim == 3:
        b, w, h = ts.shape
    else:
        b, w = ts.shape; h = 1
    out = model(ts.view(b, h, w))
    return out[0] if isinstance(out, (list, tuple)) else out

train_files = get_pln_train_files()
train_ds = None
for fname in train_files:
    ds = PLN(fname, train=True, transform=val_transform, sanomaly=sanomaly)
    train_ds = ds if train_ds is None else (train_ds.concat_ds(ds) or train_ds)

train_dl = torch.utils.data.DataLoader(
    train_ds, batch_size=p['batch_size'], shuffle=False,
    num_workers=p['num_workers'], collate_fn=collate_custom, drop_last=False)

all_preds = []
with torch.no_grad():
    for batch in train_dl:
        ts = batch['ts_org'].to(device)
        preds = torch.argmax(get_logits(model, ts), dim=1).cpu().numpy()
        all_preds.append(preds)
all_preds = np.concatenate(all_preds)
values, counts = np.unique(all_preds, return_counts=True)
normal_label = int(values[np.argmax(counts)])
print(f'Normal label (majority): {normal_label}  ({dict(zip(values.tolist(), counts.tolist()))})')

In [ ]:
def aggregate_scores(scores, method='mean'):
    if method == 'mean':  return float(np.mean(scores))
    if method == 'max':   return float(np.max(scores))
    if method == 'pct95': return float(np.percentile(scores, 95))
    return float(np.mean(scores))

# --- Inferensi per file ---
test_files  = get_pln_test_files()
rows        = []   # per-file
window_data = {}   # per-window, per file

with torch.no_grad():
    for fname in test_files:
        ds = PLN(fname, train=False, transform=val_transform, sanomaly=sanomaly)
        dl = torch.utils.data.DataLoader(
            ds, batch_size=p['batch_size'], shuffle=False,
            num_workers=p['num_workers'], collate_fn=collate_custom, drop_last=False)

        win_scores, win_labels = [], []
        raw_windows = []
        for batch in dl:
            ts     = batch['ts_org'].to(device)
            logits = get_logits(model, ts)
            probs  = F.softmax(logits, dim=1).cpu().numpy()
            scores = 1.0 - probs[:, normal_label]
            win_scores.extend(scores.tolist())
            win_labels.extend(batch['target'].cpu().numpy().tolist())
            raw_windows.append(batch['ts_org'].cpu().numpy())

        win_scores = np.array(win_scores)
        true_label = int(win_labels[0])
        file_score = aggregate_scores(win_scores, AGG)

        rows.append({
            'fname':      fname,
            'true_label': true_label,
            'score':      file_score,
            'n_windows':  len(win_scores),
            'score_min':  float(win_scores.min()),
            'score_max':  float(win_scores.max()),
            'score_std':  float(win_scores.std()),
        })
        window_data[fname] = {
            'win_scores':  win_scores,
            'raw_windows': np.concatenate(raw_windows, axis=0),  # (N_win, T, F)
            'true_label':  true_label,
        }

df = pd.DataFrame(rows)
print(f'Total file test: {len(df)}  (normal: {(df.true_label==0).sum()}, anomali: {(df.true_label==1).sum()})')
df.head(10)

In [ ]:
# --- Hitung threshold & label prediksi ---
from sklearn.metrics import precision_recall_curve

file_scores = df['score'].values
file_labels = df['true_label'].values

prec_curve, rec_curve, thr_pr = precision_recall_curve(file_labels, file_scores)
f1_curve = np.where(
    (prec_curve + rec_curve) > 0,
    2 * prec_curve * rec_curve / (prec_curve + rec_curve), 0.0)
best_idx  = int(np.argmax(f1_curve[:-1]))
THRESHOLD = float(thr_pr[best_idx])

df['pred_label'] = (df['score'] >= THRESHOLD).astype(int)
df['result'] = df.apply(lambda r: 
    'TP' if r.true_label==1 and r.pred_label==1 else
    'TN' if r.true_label==0 and r.pred_label==0 else
    'FP' if r.true_label==0 and r.pred_label==1 else 'FN', axis=1)

auroc = roc_auc_score(file_labels, file_scores) if len(np.unique(file_labels)) > 1 else float('nan')
aupr  = average_precision_score(file_labels, file_scores)
f1    = f1_score(file_labels, df['pred_label'].values, zero_division=0)

print(f'Threshold : {THRESHOLD:.4f}')
print(f'AUROC     : {auroc:.4f}')
print(f'AUPR      : {aupr:.4f}')
print(f'F1 anomaly: {f1:.4f}')
print(f"\nDistribusi: {df['result'].value_counts().to_dict()}")

## 3. Score Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = {'Normal (0)': '#2196F3', 'Anomali (1)': '#F44336'}

# Histogram
ax = axes[0]
for lbl, col, name in [(0, '#2196F3', 'Normal (0)'), (1, '#F44336', 'Anomali (1)')]:
    scores_sub = df[df.true_label == lbl]['score'].values
    ax.hist(scores_sub, bins=20, alpha=0.65, color=col, label=f'{name}  n={len(scores_sub)}')
ax.axvline(THRESHOLD, color='black', linestyle='--', linewidth=1.5, label=f'Threshold={THRESHOLD:.3f}')
ax.set_xlabel('Anomaly Score')
ax.set_ylabel('Jumlah File')
ax.set_title('Distribusi Score: Normal vs Anomali')
ax.legend()

# Swarm / strip plot per file
ax2 = axes[1]
result_colors = {'TP': '#4CAF50', 'TN': '#2196F3', 'FP': '#FF9800', 'FN': '#F44336'}
jitter = np.random.default_rng(42)
for res, col in result_colors.items():
    sub = df[df.result == res]
    if len(sub) == 0: continue
    x = jitter.uniform(-0.15, 0.15, len(sub)) + (0 if sub.true_label.iloc[0] == 0 else 1)
    ax2.scatter(x, sub['score'].values, color=col, alpha=0.8, s=60, label=res, zorder=3)
ax2.axhline(THRESHOLD, color='black', linestyle='--', linewidth=1.5, label=f'Threshold={THRESHOLD:.3f}')
ax2.set_xticks([0, 1]); ax2.set_xticklabels(['Normal (0)', 'Anomali (1)'])
ax2.set_ylabel('Anomaly Score')
ax2.set_title('Score per File (warna = hasil prediksi)')
ax2.legend()

plt.tight_layout()
out_path = BASE_DIR / 'results' / 'figures' / f'error_pln_score_dist_{AGG}.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path}')

## 4. ROC & PR Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ROC
ax = axes[0]
if not np.isnan(auroc):
    fpr, tpr, thr_roc = roc_curve(file_labels, file_scores)
    ax.plot(fpr, tpr, color='#2196F3', lw=2, label=f'AUROC = {auroc:.4f}')
    # titik threshold terpilih
    thr_idx = np.searchsorted(thr_roc[::-1], THRESHOLD)
    thr_idx = max(0, len(thr_roc) - 1 - thr_idx)
    ax.scatter([fpr[thr_idx]], [tpr[thr_idx]], color='red', zorder=5, s=80,
               label=f'Threshold={THRESHOLD:.3f}')
ax.plot([0,1],[0,1],'k--',alpha=0.4)
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve'); ax.legend()

# PR
ax2 = axes[1]
ax2.plot(rec_curve, prec_curve, color='#F44336', lw=2, label=f'AUPR = {aupr:.4f}')
# titik threshold terpilih
ax2.scatter([rec_curve[best_idx]], [prec_curve[best_idx]], color='black', zorder=5, s=80,
            label=f'Best F1={f1:.3f} @ thr={THRESHOLD:.3f}')
ax2.set_xlabel('Recall'); ax2.set_ylabel('Precision')
ax2.set_title('Precision-Recall Curve'); ax2.legend()

plt.tight_layout()
out_path = BASE_DIR / 'results' / 'figures' / f'error_pln_roc_pr_{AGG}.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path}')

## 5. Confusion Matrix

In [ ]:
cm = confusion_matrix(file_labels, df['pred_label'].values, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=False, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Pred Normal', 'Pred Anomali'],
            yticklabels=['True Normal', 'True Anomali'],
            linewidths=1, linecolor='white')

# Anotasi manual dengan warna berbeda
cell_colors = [['#2196F3', '#FF9800'], ['#F44336', '#4CAF50']]
cell_labels = [[f'TN\n{tn}', f'FP\n{fp}'], [f'FN\n{fn}', f'TP\n{tp}']]
for i in range(2):
    for j in range(2):
        ax.text(j + 0.5, i + 0.5, cell_labels[i][j],
                ha='center', va='center', fontsize=18, fontweight='bold',
                color='white')

ax.set_title(f'Confusion Matrix PLN  (threshold={THRESHOLD:.3f}, agg={AGG})', fontsize=12)
plt.tight_layout()
out_path = BASE_DIR / 'results' / 'figures' / f'error_pln_confusion_{AGG}.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'TN={tn}  FP={fp}  FN={fn}  TP={tp}')
print(f'Saved: {out_path}')

## 6. Tabel FP / FN Per File

In [ ]:
cols_show = ['fname', 'true_label', 'pred_label', 'score', 'score_min', 'score_max', 'score_std', 'n_windows', 'result']

print('=== FALSE POSITIVE (normal diprediksi anomali) ===')
fp_df = df[df.result == 'FP'][cols_show].sort_values('score', ascending=False)
display(fp_df)

print('\n=== FALSE NEGATIVE (anomali lolos / tidak terdeteksi) ===')
fn_df = df[df.result == 'FN'][cols_show].sort_values('score', ascending=True)
display(fn_df)

print('\n=== SEMUA FILE (diurutkan berdasarkan score) ===')
def highlight_result(row):
    color_map = {'TP': 'background-color: #c8e6c9',
                 'TN': 'background-color: #bbdefb',
                 'FP': 'background-color: #ffe0b2',
                 'FN': 'background-color: #ffcdd2'}
    return [color_map.get(row.result, '')] * len(row)

display(df[cols_show].sort_values('score', ascending=False)
        .style.apply(highlight_result, axis=1)
        .format({'score': '{:.4f}', 'score_min': '{:.4f}',
                 'score_max': '{:.4f}', 'score_std': '{:.4f}'}))

## 7. Time Series — False Positive Files

File **normal** yang diprediksi anomali. Cari pola apa yang membuat model bingung.

In [ ]:
def plot_file_timeseries(fname, result_label, max_features=4, figsize=(14, 3)):
    """Plot time series lengkap file + overlay skor anomali per window."""
    data = window_data[fname]
    win_scores = data['win_scores']    # (N_win,)
    raw_ts = data['raw_windows']       # (N_win, T, F)

    # Rekonstruksi time series lengkap dari windows (ambil window pertama + tail windows berikutnya)
    # Ini approximate — pakai raw .npy untuk versi tepat
    npy_path = BASE_DIR / 'datasets' / 'PLN' / 'processed' / 'data' / fname
    if not npy_path.exists():
        npy_path = npy_path.with_suffix('.npy') if not str(npy_path).endswith('.npy') else npy_path
    if npy_path.exists():
        full_ts = np.load(str(npy_path))  # (T_total, F)
    else:
        full_ts = raw_ts[0]  # fallback: window pertama saja

    n_features = min(full_ts.shape[1] if full_ts.ndim > 1 else 1, max_features)
    fig, axes = plt.subplots(n_features + 1, 1, figsize=(figsize[0], figsize[1] * (n_features + 1)),
                              sharex=False)
    fig.suptitle(f'{result_label}: {fname}  (score={df.loc[df.fname==fname, "score"].values[0]:.4f})',
                 fontsize=12, fontweight='bold', color={
                     'FP': '#FF9800', 'FN': '#F44336', 'TP': '#4CAF50', 'TN': '#2196F3'}.get(result_label, 'black'))

    # Panel atas: skor anomali per window
    ax0 = axes[0]
    ax0.bar(range(len(win_scores)), win_scores, color=[
        '#F44336' if s >= THRESHOLD else '#2196F3' for s in win_scores], width=1.0, alpha=0.8)
    ax0.axhline(THRESHOLD, color='black', linestyle='--', linewidth=1.2, label=f'Threshold={THRESHOLD:.3f}')
    ax0.set_ylabel('Score per window')
    ax0.set_xlabel('Window index')
    ax0.set_title('Anomaly score per window')
    ax0.legend(fontsize=8)

    # Panel bawah: fitur time series
    T = full_ts.shape[0]
    t_axis = np.arange(T)
    feat_names = [f'Fitur {i+1}' for i in range(n_features)]

    for i in range(n_features):
        ax = axes[i + 1]
        col = full_ts[:, i] if full_ts.ndim > 1 else full_ts
        ax.plot(t_axis, col, color='#333333', linewidth=0.8)
        ax.set_ylabel(feat_names[i])
        if i == n_features - 1:
            ax.set_xlabel('Timestep')

    plt.tight_layout()
    out_path = BASE_DIR / 'results' / 'figures' / f'error_pln_{result_label}_{fname[:20]}.png'
    plt.savefig(out_path, dpi=120, bbox_inches='tight')
    plt.show()
    plt.close()


fp_files = df[df.result == 'FP']['fname'].tolist()
if fp_files:
    print(f'FP files ({len(fp_files)}): {fp_files}')
    for fname in fp_files[:5]:  # batasi 5 file
        plot_file_timeseries(fname, 'FP')
else:
    print('Tidak ada False Positive — model sempurna untuk kelas normal.')

## 8. Time Series — False Negative Files

File **anomali** yang lolos (diprediksi normal). Ini yang paling kritis untuk deteksi pencurian listrik.

In [ ]:
fn_files = df[df.result == 'FN']['fname'].tolist()
if fn_files:
    print(f'FN files ({len(fn_files)}): {fn_files}')
    for fname in fn_files[:5]:  # batasi 5 file
        plot_file_timeseries(fname, 'FN')
else:
    print('Tidak ada False Negative — semua pencuri terdeteksi!')

## 9. Per-Window Score Heatmap

Pilih satu file spesifik, lihat skor anomali setiap window secara detail.
Berguna untuk memahami **bagian mana dalam time series** yang model anggap mencurigakan.

In [ ]:
# Ganti fname_target ke file yang ingin diinspeksi
# Contoh: ambil FN pertama, atau FP pertama
candidates = fn_files + fp_files
fname_target = candidates[0] if candidates else df['fname'].iloc[0]

print(f'Inspeksi file: {fname_target}')
print(f'True label: {window_data[fname_target]["true_label"]}  |  Pred label: {df.loc[df.fname==fname_target, "pred_label"].values[0]}  |  Result: {df.loc[df.fname==fname_target, "result"].values[0]}')

wscores = window_data[fname_target]['win_scores']   # (N_win,)
N_win = len(wscores)

fig, axes = plt.subplots(2, 1, figsize=(14, 5))

# Line plot skor per window
ax = axes[0]
ax.plot(wscores, color='#333333', linewidth=1)
ax.fill_between(range(N_win), wscores, THRESHOLD,
                where=(wscores >= THRESHOLD), color='#F44336', alpha=0.5, label='Above threshold')
ax.fill_between(range(N_win), wscores, THRESHOLD,
                where=(wscores < THRESHOLD), color='#2196F3', alpha=0.3, label='Below threshold')
ax.axhline(THRESHOLD, color='black', linestyle='--', linewidth=1.2, label=f'Threshold={THRESHOLD:.3f}')
ax.set_xlabel('Window index'); ax.set_ylabel('Anomaly score')
ax.set_title(f'Per-window score: {fname_target}'); ax.legend(fontsize=8)

# Heatmap skor — 1 baris, N_win kolom (reshaped supaya lebih mudah dibaca)
ax2 = axes[1]
cols = min(N_win, 50)                          # maks 50 kolom per baris
rows_needed = int(np.ceil(N_win / cols))
padded = np.pad(wscores, (0, rows_needed * cols - N_win), constant_values=np.nan)
heatmap_data = padded.reshape(rows_needed, cols)

im = ax2.imshow(heatmap_data, aspect='auto', cmap='RdYlGn_r', vmin=0, vmax=1)
plt.colorbar(im, ax=ax2, label='Anomaly score')
ax2.set_xlabel('Window (dalam baris)')
ax2.set_ylabel('Baris')
ax2.set_title(f'Heatmap score per window ({N_win} windows, {cols} per baris)')

plt.tight_layout()
out_path = BASE_DIR / 'results' / 'figures' / f'error_pln_heatmap_{fname_target[:20]}.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path}')

## 10. Ringkasan Error Analysis

In [ ]:
print('=' * 55)
print(f'  ERROR ANALYSIS — PLN  (agg={AGG})')
print('=' * 55)
print(f'  Total file test : {len(df)}')
print(f'  Normal          : {(df.true_label==0).sum()}')
print(f'  Anomali         : {(df.true_label==1).sum()}')
print(f'  Threshold       : {THRESHOLD:.4f}  (auto: max F1 pada PR curve)')
print(f'  AUROC           : {auroc:.4f}')
print(f'  AUPR            : {aupr:.4f}')
print(f'  F1 anomaly      : {f1:.4f}')
print(f'  TP={tp}  TN={tn}  FP={fp}  FN={fn}')
print('-' * 55)

if fp > 0:
    print(f'  FP ({fp} file) — normal diprediksi anomali:')
    for _, row in fp_df.iterrows():
        print(f'    {row.fname}  score={row.score:.4f}  std={row.score_std:.4f}')

if fn > 0:
    print(f'  FN ({fn} file) — pencuri yang lolos (tidak terdeteksi):')
    for _, row in fn_df.iterrows():
        print(f'    {row.fname}  score={row.score:.4f}  std={row.score_std:.4f}')

print('=' * 55)
print('Interpretasi FN: pencuri dengan pola mirip normal (score_std rendah = konsisten rendah).')
print('Interpretasi FP: pelanggan normal dengan pola tidak biasa atau variasi tinggi.')